# Sales Forecast API: Feature Engineering

## 1. Objective

The purpose of this notebook is to transform the raw data into a dataset suitable for machine learning.

The transformations performed here are designed not only to improve model performance, but also to ensure that every feature can be reproduced during inference.

Since this project aims to deploy a prediction API, only information available at prediction time will be used.

In [ ]:
import warnings
warnings.filterwarnings("ignore")

import pandas as pd

from src import RAW_DATA_DIR

## 2. Load Dataset

In [ ]:
train = pd.read_csv(
    RAW_DATA_DIR / "train.csv",
    parse_dates=["Date"]
)

store = pd.read_csv(
    RAW_DATA_DIR / "store.csv"
)

df = train.merge(
    store,
    on="Store",
    how="left"
)

## 3. Remove closed stores

Our objective is to predict demand when stores are operating normally. Since closed stores always report zero sales, keeping them would mostly teach the model that ``` Open = 0 → Sales = 0``` instead of learning actual sales patterns.

Therefore, only observations where the store is open are kept, and the column is subsequently dropped.

In [ ]:
df = df[df["Open"] == 1].copy()

df = df.drop(columns="Open")

## 4. Handle missing values

The missing values found in the Rossmann dataset mostly represent business situations rather than data quality problems.

For example:

- Stores without `Promo2` naturally have missing promotion dates.
- Some competitors exist but their opening date is unknown.

The chosen imputations preserve this meaning while allowing machine learning models to process the data.

### 4.1 Competition distance

Only a very small percentage of stores have missing competition distance.

The missing values are replaced with the median distance to minimize their impact.

In [ ]:
df["CompetitionDistance"] = (
    df["CompetitionDistance"]
      .fillna(df["CompetitionDistance"].median())
)

### 4.2 Promotion History

Stores that do not participate in the continuous promotion program (`Promo2`) naturally have missing values describing that promotion.

Instead of treating these values as missing information, they are replaced with sentinel values representing "no promotion".

In [ ]:
df["PromoInterval"] = (
    df["PromoInterval"]
      .fillna("None")
)

df["Promo2SinceWeek"] = (
    df["Promo2SinceWeek"]
      .fillna(0)
)

df["Promo2SinceYear"] = (
    df["Promo2SinceYear"]
      .fillna(0)
)

### 4.3 Competition opening date

Some stores have competitors, but the exact opening date is unknown.

These missing values are replaced with zero, allowing tree-based models to distinguish them from valid dates.

In [ ]:
df["CompetitionOpenSinceMonth"] = (
    df["CompetitionOpenSinceMonth"]
      .fillna(0)
)

df["CompetitionOpenSinceYear"] = (
    df["CompetitionOpenSinceYear"]
      .fillna(0)
)

## 5. Inspect data types

Before performing feature engineering, we inspect the inferred data types.

Although pandas usually infers the correct types automatically, CSV files may contain inconsistent values that lead to mixed data types. Standardizing these columns early helps prevent issues during preprocessing, model training and data serialization.

In [ ]:
df.info()

In [ ]:
for col in df.columns:
    n_types = df[col].map(type).nunique()
    if n_types > 1:
        print(f"{col}: mixed types detected")

### 5.1 Standardize categorical values

The inspection detected mixed data types in the `StateHoliday` column.

This column represents a categorical variable with the following possible values:

- `0`: No holiday
- `a`: Public holiday
- `b`: Easter holiday
- `c`: Christmas

However, the value `0` appears both as an integer (`0`) and as a string (`"0"`), resulting in inconsistent object types. Although this usually does not affect exploratory analysis, it may cause problems during serialization (e.g., Parquet) and categorical preprocessing.

To ensure consistency throughout the project, the column is explicitly converted to the string data type.

In [ ]:
df["StateHoliday"] = df["StateHoliday"].astype(str)

# Verify that no mixed-type columns remain
for col in df.columns:
    n_types = df[col].map(type).nunique()
    if n_types > 1:
        print(f"{col}: mixed types detected")

The dataset is now ready for feature engineering with consistent data types across all columns.

## 6. Calendar features

Dates are rarely useful in their raw form.

Instead, the date is decomposed into multiple features that allow the model to capture seasonal patterns such as:

- monthly seasonality
- weekly seasonality
- holidays
- quarter effects
- weekend behavior

In [ ]:
df["Year"] = df["Date"].dt.year
df["Month"] = df["Date"].dt.month
df["Day"] = df["Date"].dt.day
df["Week"] = df["Date"].dt.isocalendar().week.astype(int)
df["Quarter"] = df["Date"].dt.quarter

### 6.1 Weekend indicator

Customer behavior usually changes during weekends.

Instead of forcing the model to infer this pattern from the day of week, an explicit binary feature is created.

In [ ]:
df["IsWeekend"] = (
    df["DayOfWeek"] >= 6
).astype(int)

### 6.2 Beginning and end of month

Some businesses experience different sales behavior near the beginning or end of each month due to salary payments, billing cycles or consumer habits.

Binary indicators are created to allow the model to capture these effects.

In [ ]:
df["IsMonthStart"] = (
    df["Date"].dt.is_month_start
).astype(int)

df["IsMonthEnd"] = (
    df["Date"].dt.is_month_end
).astype(int)

## 7. Remove data leakage

The `Customers` column is highly correlated with sales.

However, the number of customers is not known when forecasting future sales. Using this variable would artificially improve model performance while making the model unusable in production.

Therefore, it is removed before training.

In [ ]:
X = df.drop(columns=["Sales", "Customers"])
y = df["Sales"]

## 8. Identify feature types

Separating numerical and categorical features simplifies the construction of the preprocessing pipeline in the next notebook.

The preprocessing pipeline will automatically apply the appropriate transformations to each feature type.

In [ ]:
categorical_features = X.select_dtypes(include=["object"]).columns.tolist()

numeric_features = X.select_dtypes(
    include=["int64", "float64", "int32"]
).columns.tolist()

print("Categorical:", categorical_features)
print("Numeric:", numeric_features)

## 9. Save processed dataset

The processed features are saved so that the modeling notebook can focus exclusively on machine learning rather than data preparation.

Separating preprocessing from model training also makes experimentation easier and improves reproducibility.

In [ ]:
from src.config import PROCESSED_DATA_DIR

X.to_parquet(PROCESSED_DATA_DIR / "X.parquet", index=False)

y.to_frame().to_parquet(
    PROCESSED_DATA_DIR / "y.parquet",
    index=False
)

## 10. Conclusions

At this stage, the dataset is ready for machine learning.

The main preprocessing decisions were:

- merged transactional and store information;
- removed observations where stores were closed;
- handled missing values using business-aware imputations;
- extracted calendar-based features from the date;
- removed variables that would introduce data leakage;
- prepared the dataset for a reproducible preprocessing pipeline.

The next notebook will focus on training and evaluating machine learning models using this processed dataset.